##Import

In [1]:
# ----------------------------------------------------------------------------------
# 環境設置與函式庫導入
# ----------------------------------------------------------------------------------
!pip install pygsheets

import sys
from google.colab import drive
from google.colab import userdata # 導入 userdata 模組
import pandas as pd
import gspread
import pygsheets
from pygsheets import Cell

# ----------------------------------------------------------------------------------
# API 金鑰與 ID 安全讀取 (使用 userdata.get())
# ----------------------------------------------------------------------------------

YOUR_SERVICE_FILE_PATH = userdata.get("SERVICE_FILE_PATH")
TPR_RSV_ID = userdata.get('TPR_RSV_ID')
CBMS_ID = userdata.get('CBMS_ID')
FOMS_ID = userdata.get('FOMS_ID')
INPUT_FILE_ID = userdata.get('INPUT_FILE_ID')

if not all([YOUR_SERVICE_FILE_PATH, TPR_RSV_ID, CBMS_ID, FOMS_ID, INPUT_FILE_ID]):
    print("FATAL ERROR: 至少一個 Sheet ID 或服務金鑰路徑未從 Secrets Manager 中讀取成功。")
    sys.exit(1)


# ----------------------------------------------------------------------------------
# 服務授權與檔案載入
# ----------------------------------------------------------------------------------

# 掛載 Google Drive
try:
    drive.mount('/content/gdrive')
except Exception as e:
    print(f"FATAL ERROR: Drive 掛載失敗。錯誤: {e.__class__.__name__}")
    sys.exit(1)

# 授權 pygsheets
try:
    gc = pygsheets.authorize(service_file=YOUR_SERVICE_FILE_PATH)
    print("✅ API 授權與環境設置成功。")
except Exception as e:
    print(f"FATAL ERROR: 授權失敗。錯誤: {e}")
    sys.exit(1)

# --- 全域工作表物件聲明 ---
global INPUT_SHEET, FO_INPUT_SHEET, TPR_restaurant, TPR_RSVPlans, TW_CBMS, CBMS_master, PAYMENT_ACCOUNT_DEV_SHEET, TD_master, FO_MS

try:
    # 讀取輸入/來源檔案 (INPUT_FILE_ID)
    change_master_sheet = gc.open_by_key(INPUT_FILE_ID)
    INPUT_SHEET = change_master_sheet.worksheet_by_title("TMS_input")
    FO_INPUT_SHEET = change_master_sheet.worksheet_by_title("FO_input")
    INPUT_pending = change_master_sheet.worksheet_by_title("TMS_input_pending")
    FO_INPUT_pending = change_master_sheet.worksheet_by_title("FO_input_pending")

    # 讀取 TPR 主表 (TPR_RSV_ID)
    TPR_master = gc.open_by_key(TPR_RSV_ID)
    TPR_restaurant = TPR_master.worksheet_by_title("[New] TW RSV Restaurants")
    TPR_RSVPlans = TPR_master.worksheet_by_title("TW RSV Plans")

    # 讀取 CBMS 主表 (CBMS_ID)
    CBMS_master = gc.open_by_key(CBMS_ID)
    TW_CBMS = CBMS_master.worksheet_by_title("TW")
    PAYMENT_ACCOUNT_DEV_SHEET = CBMS_master.worksheet_by_title("虛擬帳號 for Dev")

    # 讀取 Takeout 主表 (FOMS_ID)
    TD_master = gc.open_by_key(FOMS_ID)
    FO_MS = TD_master.worksheet_by_title("TO&FD-TW")

    print("所有檔案和工作表物件已成功載入並在全域範圍可用。")

except Exception as e:
    print(f"FATAL ERROR: 無法開啟或定位工作表。錯誤: {e.__class__.__name__} - {e}")
    sys.exit(1)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 kB 2.1 MB/s eta 0:00:00
Mounted at /content/gdrive
✅ API 授權與環境設置成功。
所有檔案和工作表物件已成功載入並在全域範圍可用。


In [2]:
SKIP_KEYWORDS = ['不需變更公司統編', '不需變更公司抬頭', '不需變更餐廳名稱', '不需變更合約寄送地址',
                 '不需變更帳務聯絡人姓名', '不需變更帳務聯絡人 Email', '不需變更帳務聯絡人電話',
                 '', 'N/A', 'na', '不需變更', None]

In [8]:
def parse_input_row(row_data):
    row_data = row_data[:15]
    row_data += [''] * (15 - len(row_data))

    return {
        'id': str(row_data[0] or '').strip(),
        'old_name': str(row_data[1] or '').strip(),
        'new_title': str(row_data[2] or '').strip(),
        'new_tax': str(row_data[3] or '').strip(),
        'new_name': str(row_data[4]  or '').strip(),
        'new_billing_name': str(row_data[5] or '').strip(),
        'new_billing_phone': str(row_data[6] or '').strip(),
        'new_billing_email': str(row_data[7] or '').strip(),
        'new_billing_address': str(row_data[8] or '').strip(),
        'payment_account': str(row_data[11] or '').strip(),
        'contract': str(row_data[14] or '').strip()
    }

# --- 解析 INPUT SHEET 並區分有效行 ---
rows = INPUT_SHEET.get_all_values()

success_rows = []

success_rows = [
    (i, parse_input_row(row))  # （列數, 解析後的列資料）
    for i, row in enumerate(rows[2:], start=3)  # 從 INPUT SHEET 的第3行開始
    if row[0].strip() # 只抓 Branch ID 不為空的列
]

for sheet_row_num, parsed in success_rows:
    branch_id = parsed['id']
    old_name = parsed['old_name']

    print(
    f"Sheet Row {sheet_row_num} | "
    f"{branch_id} | "
    f"Name: {old_name}"
    )

Sheet Row 3 | -KoqkE0zNpSo2w4W16pJ | Name: 漉海鮮蒸氣鍋 松江店
Sheet Row 4 | -KoqkE2nwlsxERgBOMcP | Name: 漉海鮮蒸氣鍋 南港中信店
Sheet Row 5 | -LUcE1z7aIhY19gdRpXv | Name: 漉海鮮蒸氣鍋 汐止遠雄店
Sheet Row 6 | -MEKcgiOJqFwjkWr17Sq | Name: 海宴 新台菜會館
Sheet Row 7 | -LKtNkF-L1ZnPpDXoR1o | Name: 藝奇 新日本料理 高雄夢時代店
Sheet Row 8 | -MaDBNKmo75AHXLrRA2Q | Name: 藝奇 新日本料理 台北衡陽店
Sheet Row 9 | -MaDC8g3_nfAqlYzQDp4 | Name: 藝奇 新日本料理 桃園南華店
Sheet Row 10 | -MaDE63-Ol1akQ3zxbcY | Name: 藝奇 新日本料理 竹北光明店
Sheet Row 11 | -MaDFZWrADFULUfJGW1W | Name: 藝奇 新日本料理 嘉義耐斯店
Sheet Row 12 | -MaDG4Po9PxuWq9D-LEy | Name: 藝奇 新日本料理 台中大墩店
Sheet Row 13 | -NGMZaCKeFVAo7vdAUwf | Name: 藝奇 日本料理岩板燒 台南南紡店
Sheet Row 14 | -NVSdsUrrvM1YOMkqM_6 | Name: 藝奇日本料理岩板燒-台中國安店
Sheet Row 15 | -O80fjK-iKJoXtLRZKqI | Name: 藝奇 日本料理岩板燒 台北南京東店


In [4]:
def FO_parse_input_row(row_data):
    row_data = row_data[:21]
    row_data += [''] * (21 - len(row_data))

    return {
        'id': str(row_data[0] or '').strip(),                           # A: Branch ID
        'old_name': str(row_data[1] or '').strip(),                     # B: 申請變更之餐廳名稱(舊)
        'new_title': str(row_data[2] or '').strip(),                    # C: 變更後之公司抬頭
        'new_tax': str(row_data[3] or '').strip(),                      # D: 變更後之公司統編
        'new_name': str(row_data[4] or '').strip(),                     # E: 變更後之餐廳名稱(新)
        'new_boss': str(row_data[5] or '').strip(),                     # F: 變更後之公司負責人
        'new_address': str(row_data[6] or '').strip(),                  # G: 變更後之公司登記地址

        # 匯款帳戶
        'fo_bank_name': str(row_data[7] or '').strip(),                 # H
        'fo_bank_code': str(row_data[8] or '').strip(),                 # I
        'fo_branch_name': str(row_data[9] or '').strip(),               # J
        'fo_branch_code': str(row_data[10] or '').strip(),              # K
        'fo_account_name': str(row_data[11] or '').strip(),             # L
        'fo_account_number': str(row_data[12] or '').strip(),           # M

        # 帳務資訊
        'new_billing_name': str(row_data[13] or '').strip(),            # N
        'new_billing_phone': str(row_data[14] or '').strip(),           # O
        'new_billing_email': str(row_data[15] or '').strip(),           # P
        'new_billing_address': str(row_data[16] or '').strip(),         # Q

        'contract': str(row_data[20] or '').strip(),                    # U
    }


# --- 解析 FO INPUT SHEET 並區分有效行 ---
FO_rows = FO_INPUT_SHEET.get_all_values()

FO_success_rows = []

FO_success_rows = [
    (i, FO_parse_input_row(row))  # （列數, 解析後的列資料）
    for i, row in enumerate(FO_rows[2:], start=3)  # 從 FO INPUT SHEET 的第3行開始
    if row[0].strip() # 只抓 Branch ID 不為空的列
]

for sheet_row_num, parsed in FO_success_rows:
    branch_id = parsed['id']
    old_name = parsed['old_name']

    print(
    f"Sheet Row {sheet_row_num} | "
    f"{branch_id} | "
    f"Name: {old_name}"
    )

Sheet Row 3 | -LKtNkF-L1ZnPpDXoR1o | Name: 藝奇 新日本料理 高雄夢時代店
Sheet Row 4 | -MaDBNKmo75AHXLrRA2Q | Name: 藝奇 新日本料理 台北衡陽店
Sheet Row 5 | -MaDC8g3_nfAqlYzQDp4 | Name: 藝奇 新日本料理 桃園南華店
Sheet Row 6 | -MaDE63-Ol1akQ3zxbcY | Name: 藝奇 新日本料理 竹北光明店
Sheet Row 7 | -MaDFZWrADFULUfJGW1W | Name: 藝奇 新日本料理 嘉義耐斯店
Sheet Row 8 | -MaDG4Po9PxuWq9D-LEy | Name: 藝奇 新日本料理 台中大墩店
Sheet Row 9 | -NGMZaCKeFVAo7vdAUwf | Name: 藝奇 日本料理岩板燒 台南南紡店
Sheet Row 10 | -NVSdsUrrvM1YOMkqM_6 | Name: 藝奇日本料理岩板燒-台中國安店
Sheet Row 11 | -O80fjK-iKJoXtLRZKqI | Name: 藝奇 日本料理岩板燒 台北南京東店


##TPR

In [9]:
# ====================================================
# Step 1: 讀 TW RSV Restaurants 現有資料與 INPUT_SHEET
# ====================================================
input_records = INPUT_SHEET.get_all_records(head=2)
try:
    tpr_ids = TPR_restaurant.get_col(1, include_tailing_empty=False) # A 欄
    tpr_status_j = TPR_restaurant.get_col(10, include_tailing_empty=False) # J 欄
except Exception as e:
    print(f"FATAL ERROR: Failed to pre-load TPR Master columns: {e}")
    sys.exit(1)

branch_row_map = {}
num_status_rows = len(tpr_status_j)
for idx, branch_id in enumerate(tpr_ids, start=1):
    status_list_index = idx - 1  # 計算 Python 列表索引 (0-based)
    if status_list_index < num_status_rows:
        status_j = tpr_status_j[status_list_index].strip()
    else:
        status_j = ''
    if status_j == '':
        branch_row_map[branch_id] = idx # 儲存 1-based 行號

# ============================
# Step 2: 定義更新函式
# ============================
def tpr_updates(row, branch_row_map):
    bid = row['id']
    insert_rows_list = []
    restaurant_update_cells = []
    rsv_update_cells = []
    tpr_pending = []
    status_log = {}

    if bid not in branch_row_map:
        status_log['Restaurants'] = 'X'
        status_log['RSV Plans'] = 'X'
        status_log['Inserted_Row'] = 'N/A'
        return insert_rows_list, restaurant_update_cells, rsv_update_cells, tpr_pending, status_log

    latest_row_idx = branch_row_map[bid]  # Excel 1-based
    latest_row_data = TPR_restaurant.get_row(latest_row_idx, include_tailing_empty=True)

    # 判斷各類變更
    has_name_change = bool(row.get('new_name')) and row.get('new_name') not in SKIP_KEYWORDS
    has_tax_change = bool(row.get('new_tax')) and row.get('new_tax') not in SKIP_KEYWORDS
    has_title_change = bool(row.get('new_title')) and row.get('new_title') not in SKIP_KEYWORDS
    has_billing_change = any(
        bool(row.get(k))
        for k in ['new_billing_address', 'new_billing_name', 'new_billing_email', 'new_billing_phone', 'payment_account']
    )

    has_tax_or_title_change = has_tax_change or has_title_change

    # 判斷合約狀態與限制情境
    contract_status = str(row.get('contract', '')).strip().upper()
    contract_not_back = contract_status != 'Y'

    # 🔹 情境二判斷: 有改統編但合約未回
    is_restricted = has_tax_change and contract_not_back

    # 🔹 情境一判斷: 無限制類
    is_unrestricted = not is_restricted

    # ============================
    # 處理 情境二(有限制類)
    # ============================
    if is_restricted:
        # tax/title 改動丟到 pending
        pending_data = [
            row.get('id'),
            row.get('old_name'),
            row.get('new_title'),
            row.get('new_tax'),
            row.get('payment_account')
        ]
        tpr_pending.append(pending_data)

        # name 改動允許下加一列
        if has_name_change:
            TPR_restaurant.update_value((latest_row_idx, 10), '資料變更')
            new_row = list(latest_row_data)
            new_row[1] = row['new_name']  # 更新名稱

            # billing 改動只發生在新列(最終顯示列)
            if has_billing_change:
                if row.get('payment_account') and row.get('payment_account') not in SKIP_KEYWORDS:
                    new_row[8] = row['payment_account']
                if row.get('new_billing_address') and row.get('new_billing_address') not in SKIP_KEYWORDS:
                    new_row[10] = row['new_billing_address']
                if row.get('new_billing_name') and row.get('new_billing_name') not in SKIP_KEYWORDS:
                    new_row[11] = row['new_billing_name']
                if row.get('new_billing_email') and row.get('new_billing_email') not in SKIP_KEYWORDS:
                    new_row[12] = row['new_billing_email']
                if row.get('new_billing_phone') and row.get('new_billing_phone') not in SKIP_KEYWORDS:
                    new_row[13] = row['new_billing_phone']

            insert_at = latest_row_idx
            insert_rows_list.append((insert_at, new_row))
            branch_row_map[bid] = insert_at
            status_log['Restaurants'] = 'H'  # H表示有限制但處理了name
            status_log['Inserted_Row'] = 'Y'

        # 只有 billing 改動(無name改動) → 允許直接覆蓋原始列
        elif has_billing_change and not has_name_change:
            BILLING_UPDATE_MAP = {
                'new_billing_address': 11,
                'new_billing_name': 12,
                'new_billing_email': 13,
                'new_billing_phone': 14,
                'payment_account': 9
            }
            updates_collected = 0
            for input_key, output_col in BILLING_UPDATE_MAP.items():
                val = row.get(input_key)
                if val and val not in SKIP_KEYWORDS:
                    col_letter = chr(64 + output_col)
                    cell_address = f'{col_letter}{latest_row_idx}'
                    try:
                        TPR_restaurant.update_value(cell_address, val)
                        updates_collected += 1
                    except Exception as e:
                        print(f"Error updating cell {cell_address}: {e}")

            if updates_collected > 0:
                status_log['Restaurants'] = 'H'
            status_log['Inserted_Row'] = 'N'

        else:
            # 只有tax/title改動,已丟 pending
            status_log['Restaurants'] = 'P'
            status_log['Inserted_Row'] = 'N'

    # ============================
    # 處理 情境一(無限制類)
    # ============================
    else:  # is_unrestricted
        # 若只有 billing change → 直接覆蓋當前列
        only_billing_change = has_billing_change and not (has_name_change or has_tax_or_title_change)

        if only_billing_change:
            BILLING_UPDATE_MAP = {
                'new_billing_address': 11,
                'new_billing_name': 12,
                'new_billing_email': 13,
                'new_billing_phone': 14,
                'payment_account': 9
            }
            updates_collected = 0
            for input_key, output_col in BILLING_UPDATE_MAP.items():
                val = row.get(input_key)
                if val and val not in SKIP_KEYWORDS:
                    col_letter = chr(64 + output_col)
                    cell_address = f'{col_letter}{latest_row_idx}'
                    try:
                        TPR_restaurant.update_value(cell_address, val)
                        updates_collected += 1
                    except Exception as e:
                        print(f"Error updating cell {cell_address}: {e}")

            if updates_collected > 0:
                status_log['Restaurants'] = 'OB'
            else:
                status_log['Restaurants'] = 'N'
            status_log['Inserted_Row'] = 'N'

        # 若有 has_tax/name/title_change → 下加一列
        elif has_name_change or has_tax_or_title_change:
            TPR_restaurant.update_value((latest_row_idx, 10), '資料變更')
            new_row = list(latest_row_data)

            # 更新所有變更欄位
            if has_name_change:
                new_row[1] = row['new_name']
            if has_tax_change:
                new_row[6] = row['new_tax']
            if has_title_change:
                new_row[7] = row['new_title']

            # billing 改動只發生在新列(最終顯示列)
            if has_billing_change:
                if row.get('payment_account') and row.get('payment_account') not in SKIP_KEYWORDS:
                    new_row[8] = row['payment_account']
                if row.get('new_billing_address') and row.get('new_billing_address') not in SKIP_KEYWORDS:
                    new_row[10] = row['new_billing_address']
                if row.get('new_billing_name') and row.get('new_billing_name') not in SKIP_KEYWORDS:
                    new_row[11] = row['new_billing_name']
                if row.get('new_billing_email') and row.get('new_billing_email') not in SKIP_KEYWORDS:
                    new_row[12] = row['new_billing_email']
                if row.get('new_billing_phone') and row.get('new_billing_phone') not in SKIP_KEYWORDS:
                    new_row[13] = row['new_billing_phone']

            insert_at = latest_row_idx
            insert_rows_list.append((insert_at, new_row))
            branch_row_map[bid] = insert_at
            status_log['Restaurants'] = 'Y'
            status_log['Inserted_Row'] = 'Y'

        else:
            status_log['Restaurants'] = 'N'
            status_log['Inserted_Row'] = 'N'

    # --- RSV Plans 餐廳名稱更新 ---
    if has_name_change:
        rsv_ids = [str(x).strip() for x in TPR_RSVPlans.get_col(1, include_tailing_empty=False)]
        for idx, rid in enumerate(rsv_ids, start=1):
            if rid == bid:
                rsv_update_cells.append(pygsheets.Cell((idx, 2), row['new_name']))
        status_log['RSV Plans'] = 'Y'
    else:
        status_log['RSV Plans'] = 'N'

    return insert_rows_list, restaurant_update_cells, rsv_update_cells, tpr_pending, status_log

# ============================
# Step 3: 主迴圈收集操作
# ============================
all_insert_rows = []
all_restaurant_cells = []
all_rsv_cells = []
all_tpr_pending = []
tpr_check = []

for i, (row_idx, row) in enumerate(success_rows, start=1):
    insert_rows, restaurant_cells, rsv_cells, tpr_pending, status_log = tpr_updates(row, branch_row_map)
    all_insert_rows.extend(insert_rows)
    all_restaurant_cells.extend(restaurant_cells)
    all_rsv_cells.extend(rsv_cells)
    all_tpr_pending.extend(tpr_pending)

    log_msg = (
        f"Restaurants: {status_log.get('Restaurants', 'N/A')} | "
        f"Inserted Row: {status_log.get('Inserted_Row', 'N/A')} | "
        f"RSV Plans: {status_log.get('RSV Plans', 'N/A')}  "
    )
    tpr_check.append([log_msg])

    print(f"[{i}/{len(success_rows)}] ID: {row['id']} | Status: "
          f"R: {status_log.get('Restaurants', 'N/A')}, "
          f"P: {status_log.get('RSV Plans', 'N/A')}")

# ==================
# Step 4: 批次更新
# ==================
# 由後往前插入新列
for row_idx, row_values in sorted(all_insert_rows, key=lambda x: x[0], reverse=True):
    print(f"Inserting new row at {row_idx}")
    TPR_restaurant.insert_rows(row=row_idx, number=1, values=[row_values])

try:
    if all_restaurant_cells:
        TPR_restaurant.update_cells(all_restaurant_cells)
    if all_rsv_cells:
        TPR_RSVPlans.update_cells(all_rsv_cells)
    if all_tpr_pending:
        pending_last_row = len(INPUT_pending.get_col(1, include_tailing_empty=False))
        start_row = pending_last_row + 1
        INPUT_pending.update_values(f"A{start_row}", all_tpr_pending)
        print(f"TMS_pending: 從第 {start_row} 行開始")

    start_row = 3
    INPUT_SHEET.update_values(
        (start_row, 13),  # M 欄
        tpr_check
    )
except Exception as e:
    print("更新過程發生錯誤,停止寫入檢核:", e)

print("=== All updates completed ===")

[1/13] ID: -KoqkE0zNpSo2w4W16pJ | Status: R: Y, P: N
[2/13] ID: -KoqkE2nwlsxERgBOMcP | Status: R: Y, P: N
[3/13] ID: -LUcE1z7aIhY19gdRpXv | Status: R: Y, P: N
[4/13] ID: -MEKcgiOJqFwjkWr17Sq | Status: R: Y, P: N
[5/13] ID: -LKtNkF-L1ZnPpDXoR1o | Status: R: X, P: X
[6/13] ID: -MaDBNKmo75AHXLrRA2Q | Status: R: X, P: X
[7/13] ID: -MaDC8g3_nfAqlYzQDp4 | Status: R: X, P: X
[8/13] ID: -MaDE63-Ol1akQ3zxbcY | Status: R: X, P: X
[9/13] ID: -MaDFZWrADFULUfJGW1W | Status: R: X, P: X
[10/13] ID: -MaDG4Po9PxuWq9D-LEy | Status: R: X, P: X
[11/13] ID: -NGMZaCKeFVAo7vdAUwf | Status: R: X, P: X
[12/13] ID: -NVSdsUrrvM1YOMkqM_6 | Status: R: X, P: X
[13/13] ID: -O80fjK-iKJoXtLRZKqI | Status: R: X, P: X
Inserting new row at 9
Inserting new row at 7
Inserting new row at 5
Inserting new row at 3
=== All updates completed ===


##CBMS

In [ ]:
# ============================
# Step 1: 定義虛擬帳號追蹤函式
# ============================

TRACKING_ROW_COUNTER = None

def track_payment_account_change(branch_id, new_account, original_account, restaurant_name):
    global PAYMENT_ACCOUNT_DEV_SHEET, TRACKING_ROW_COUNTER

    if TRACKING_ROW_COUNTER is None:
        try:
            existing_ids = PAYMENT_ACCOUNT_DEV_SHEET.get_col(1, include_tailing_empty=False)
            TRACKING_ROW_COUNTER = len(existing_ids) + 1
        except Exception:
            TRACKING_ROW_COUNTER = 2 # 發生錯誤時，設置為 Row 2

    next_row = TRACKING_ROW_COUNTER # 使用當前的計數器值

    data_to_output = [branch_id, restaurant_name, new_account, original_account]

    range_to_update = f'B{next_row}:E{next_row}'

    try:
        PAYMENT_ACCOUNT_DEV_SHEET.update_values(range_to_update, [data_to_output])

        TRACKING_ROW_COUNTER += 1  # 寫入成功後，遞增全局計數器
        return f"Row {next_row}"
    except Exception as e:
        print(f"Error during tracking update: {e}")
        return f"Failed: {e.__class__.__name__}"

In [ ]:
cbms_ids = [str(x).strip() for x in TW_CBMS.get_col(1, include_tailing_empty=False)]

# ============================
# Step 2: 定義更新函式
# ============================

def cbms_updates(row, cbms_sheet, cbms_ids):

    bid = row['id']
    status_log = {}
    cbms_update_cells = []

    if not bid:
        status_log['CBMS'] = 'X'
        status_log['PaymentTracking'] = 'N/A'
        return cbms_update_cells, status_log

    try:
        cbms_row_idx = cbms_ids.index(bid) + 1  # 1-based
    except ValueError:
        status_log['CBMS'] = 'X'
        status_log['PaymentTracking'] = 'N/A'
        return cbms_update_cells, status_log

    # --- 讀取原始虛擬帳戶 ---
    original_row = cbms_sheet.get_row(cbms_row_idx, include_tailing_empty=True)
    original_account_clean = str(original_row[17]).strip()  # R 欄

    status_log['original account'] = original_account_clean

    # --- 排除更新合約還沒回來的統編 ---
    contract_status = str(row.get('contract', '')).strip().upper()
    contract_not_back = contract_status != 'Y'
    pending_triggered = False

    has_tax_change = bool(row.get('new_tax')) and row.get('new_tax') not in SKIP_KEYWORDS

    if has_tax_change and contract_not_back:
      pending_triggered = True
      status_log['CBMS'] = 'P'


    # --- 欄位映射 ---
    field_map = {
        'new_name': 2, 'new_title': 16, 'new_tax': 17, 'payment_account': 18,
        'new_billing_phone': 20, 'new_billing_email': 19, 'new_billing_name': 15
    }

    updates_collected = 0

    # --- 收集非地址欄位更新指令 ---
    for key, col in field_map.items():
      if pending_triggered and key in ['new_tax', 'new_title']:
        status_log['CBMS'] = 'P'
        continue

      val_raw = row.get(key)
      val = str(val_raw).strip()
      if val and val not in SKIP_KEYWORDS:
          cbms_update_cells.append(pygsheets.Cell((cbms_row_idx, col), val))
          updates_collected += 1

    # --- 地址拆分邏輯 (單獨執行) ---
    new_billing_address = row.get('new_billing_address')
    address_val = str(new_billing_address).strip()

    if address_val and address_val not in SKIP_KEYWORDS:
        city_county = address_val[:3] # 取前三個字
        rest_address = address_val[3:].strip()

        cbms_update_cells.append(pygsheets.Cell((cbms_row_idx, 24), city_county))  # 寫入 X 欄 (縣市/直轄市) = 24
        updates_collected += 1

        cbms_update_cells.append(pygsheets.Cell((cbms_row_idx, 21), rest_address))  # 寫入 U 欄 (其餘地址) = 21
        updates_collected += 1

    if updates_collected > 0:
        if pending_triggered:
            # 有資料更新，但 Pending 觸發 -> Half-Finished
            status_log['CBMS'] = f'H ({updates_collected} cells)'
        else:
            # 有資料更新，且 Pending 未觸發 -> Done
            status_log['CBMS'] = f'Y ({updates_collected} cells)'
    else:
        if pending_triggered:
            # 無資料更新，但 Pending 觸發 -> Pending (保持 'P' 狀態)
            pass
        else:
            # 既無更新，也無 Pending 觸發 -> Skipped
            status_log['CBMS'] = 'N'

    # --- payment_account 追蹤 ---
    payment_account = str(row.get('payment_account')).strip()
    if not (payment_account and payment_account not in SKIP_KEYWORDS):
        status_log['PaymentTracking'] = 'S'
    elif payment_account != original_account_clean:
      restaurant_name = str(original_row[1]).strip()

    if payment_account and payment_account not in SKIP_KEYWORDS and payment_account != original_account_clean:
        track_result = track_payment_account_change(
            bid, payment_account, original_account_clean, restaurant_name
        )
        status_log['PaymentTracking'] = track_result
    else:
        status_log['PaymentTracking'] = 'N'

    # --- cbms_updates 輸出 ---
    return cbms_update_cells, status_log


# ============================
# Step 3: 執行更新
# ============================

all_cbms_cells = []
cbms_check = []

for i, (row_idx, row) in enumerate(success_rows, start=1):  # 拆 tuple
    cbms_cells, status_log = cbms_updates(row, TW_CBMS, cbms_ids)
    all_cbms_cells.extend(cbms_cells)

    log_msg = (
        f"CBMS: {status_log.get('CBMS', 'N/A')} | "
        f"PaymentTracking: {status_log.get('PaymentTracking', 'N/A')}"
    )
    cbms_check.append([log_msg])

    print(f"[{i}/{len(success_rows)}] ID: {row['id']} | Status: "
          f"C: {status_log.get('CBMS', 'N/A')} | "
          f"A: {status_log.get('original account')} | "
          f"T: {status_log.get('PaymentTracking', 'N/A')}"
    )

# --- 批次寫入 CBMS ---
try:
  if all_cbms_cells:
      TW_CBMS.update_cells(all_cbms_cells)
  INPUT_SHEET.update_values(  # 更新完成才寫入檢核
              (start_row, 14),  # (row, col)，N 欄
              cbms_check)
except Exception as e:
    print("更新過程發生錯誤，停止寫入檢核：", e)

print("=== All updates completed ===")

[1/4] ID: -KoqkE0zNpSo2w4W16pJ | Status: C: Y (3 cells) | A: 43344531127847 | T: Row 14
[2/4] ID: -KoqkE2nwlsxERgBOMcP | Status: C: Y (3 cells) | A: 43344531127847 | T: Row 15
[3/4] ID: -LUcE1z7aIhY19gdRpXv | Status: C: Y (3 cells) | A: 43344531127847 | T: Row 16
[4/4] ID: -MEKcgiOJqFwjkWr17Sq | Status: C: Y (3 cells) | A: 43344531127847 | T: Row 17
=== All updates completed ===


##FO

In [ ]:
# ====================================================
# 預先讀取 FO_MS 的欄位資料
# ====================================================
FO_ids_CD = [str(x).strip() for x in FO_MS.get_col(82, include_tailing_empty=True)]  # CD欄: ID
FO_status_AF = FO_MS.get_col(32, include_tailing_empty=True)  # AF欄: 狀態欄

max_len = max(len(FO_ids_CD), len(FO_status_AF))
while len(FO_ids_CD) < max_len:
    FO_ids_CD.append('')
while len(FO_status_AF) < max_len:
    FO_status_AF.append('')

# ====================================================
# FO_updates 函式（最新版）
# ====================================================
def FO_updates(row, FO_MS, FO_ids_CD, FO_status_AF):
    bid = row.get('id')
    FO_update_cells = []
    FO_pending = []
    status_log = {}

    has_name_change = bool(row.get('new_name')) and str(row.get('new_name')).strip() not in SKIP_KEYWORDS

    if not bid:
        status_log['FO'] = 'ID not found'
        return FO_update_cells, FO_pending, status_log

    # --- 定位邏輯: 在CD欄中找相同ID, 且AF欄為空的資料列 ---
    min_len = min(len(FO_ids_CD), len(FO_status_AF))
    matching_indices = [i + 1 for i in range(min_len) if FO_ids_CD[i] == bid]

    if not matching_indices:
        status_log['FO'] = 'ID not found in CD column'
        return FO_update_cells, FO_pending, status_log

    latest_row_idx = None
    all_name_update_rows = []

    for idx in matching_indices:
        list_idx = idx - 1
        raw_af_value = FO_status_AF[list_idx] if list_idx < len(FO_status_AF) else None
        af_value = str(raw_af_value).strip()
        all_name_update_rows.append(idx)

        if not af_value or af_value.upper() == 'NONE':
            latest_row_idx = idx  # 最後一筆 AF 空的即為最新行

    # ------------------ 錯誤處理 ------------------
    if latest_row_idx is None:
        first_debug_value = str(FO_status_AF[matching_indices[0] - 1]).strip()
        status_log['FO'] = f'ID found but AF column not empty. First content: [{first_debug_value}]'
        return FO_update_cells, FO_pending, status_log

    # ====================================================
    # 判斷合約邏輯
    # ====================================================
    need_contract = False
    contract_approved = str(row.get('contract', '')).strip().upper() == 'Y'

    # D 欄：統編
    if row.get('new_tax') and str(row.get('new_tax')).strip() not in SKIP_KEYWORDS:
        need_contract = True

    # H-M 欄群：銀行資訊
    for field in [
        'fo_bank_name', 'fo_bank_code', 'fo_branch_name',
        'fo_branch_code', 'fo_account_name', 'fo_account_number',
    ]:
        val = row.get(field, '')
        if val and str(val).strip() not in SKIP_KEYWORDS:
            need_contract = True
            break

    # ====================================================
    # 若需合約但未回，仍允許修改非合約欄
    # ====================================================
    pending_flag = False
    if need_contract and not contract_approved:
        pending_data = [
            row.get('id'),
            row.get('old_name', ''),
            row.get('new_title', ''),
            row.get('new_tax', ''),
            row.get('fo_bank_name', ''),
            row.get('fo_bank_code', ''),
            row.get('fo_branch_name', ''),
            row.get('fo_branch_code', ''),
            row.get('fo_account_name', ''),
            row.get('fo_account_number', ''),
        ]
        FO_pending.append(pending_data)
        pending_flag = True

    # ====================================================
    # 欄位更新
    # ====================================================
    update_fields_latest_only = {
        'new_title': 22,
        'new_boss': 23,
        'new_tax': 24,
        'new_address': 27,
        'new_billing_name': 10,
        'new_billing_email': 11,
        'new_billing_phone': 12,
        'new_billing_address': 14,
        'fo_bank_name': 68,
        'fo_branch_name': 69,
        'fo_bank_code': 70,
        'fo_branch_code': 71,
        'fo_account_name': 72,
        'fo_account_number': 73,
    }

    updates_collected = 0
    for input_key, col in update_fields_latest_only.items():
        val_raw = row.get(input_key)
        val = str(val_raw).strip()
        if val and val not in SKIP_KEYWORDS:
            # 若該欄需要合約但未通過，則不更新（留 pending）
            if need_contract and not contract_approved and input_key in ['new_tax', 'new_title', 'fo_bank_name', 'fo_bank_code', 'fo_branch_name',
        'fo_branch_code', 'fo_account_name', 'fo_account_number']:
                continue
            FO_update_cells.append(pygsheets.Cell((latest_row_idx, col), val))
            updates_collected += 1

    # 名稱可同時更新多行
    if has_name_change:
        for row_idx in all_name_update_rows:
            FO_update_cells.append(pygsheets.Cell((row_idx, 3), row['new_name']))
            updates_collected += 1

    # ====================================================
    # 狀態判定
    # ====================================================
    if pending_flag and updates_collected > 0:
        status_log['FO'] = f"H ({updates_collected} cells)"  # half-finished
    elif pending_flag:
        status_log['FO'] = "P"  # pending only
    elif updates_collected > 0:
        status_log['FO'] = f"Y ({updates_collected} cells)"
    else:
        status_log['FO'] = "N"

    return FO_update_cells, FO_pending, status_log

# ====================================================
# 主執行迴圈
# ====================================================
all_FO_cells = []
all_FO_pending = []
FO_check = []

for i, (sheet_row_num, data_dict) in enumerate(FO_success_rows, start=1):
    FO_cells, FO_pending, status_log = FO_updates(
        data_dict, FO_MS, FO_ids_CD, FO_status_AF
    )
    all_FO_cells.extend(FO_cells)
    all_FO_pending.extend(FO_pending)

    log_msg = f"FOMS: {status_log.get('FO', 'N/A')}"
    FO_check.append([log_msg])

    print(f"[{i}/{len(FO_success_rows)}] ID: {data_dict.get('id')} ｜ {status_log.get('FO', 'N/A')}")

# ====================================================
# 批次更新（不清空 pending）
# ====================================================
try:
    if all_FO_cells:
        FO_MS.update_cells(all_FO_cells)

    if all_FO_pending:
        pending_last_row = len(FO_INPUT_pending.get_col(1, include_tailing_empty=False))
        start_row = pending_last_row + 1
        FO_INPUT_pending.update_values(f"A{start_row}", all_FO_pending)
        print(f"FO_pending: 從第 {start_row} 行開始")

    start_row = 3
    FO_INPUT_SHEET.update_values((start_row, 20), FO_check)

except Exception as e:
    print("❌ 更新過程發生錯誤，停止寫入檢核：", e)

print("=== All FO updates completed ===")


[1/10] ID: -LdHEa-VbnwZFcZkixlk ｜ Y (1 cells)
[2/10] ID: -LdHFou_nfB_7tBK84Na ｜ Y (1 cells)
[3/10] ID: -MnORIMWbxmqVyCn6Kco ｜ Y (1 cells)
[4/10] ID: -NMm3rXT5hHIHv8-ravc ｜ Y (1 cells)
[5/10] ID: -L73mD45Ve5ND_OVByQW ｜ ID not found in CD column
[6/10] ID: -MtGwPlFgpTsJJ9Zl9X5 ｜ ID not found in CD column
[7/10] ID: -M0qaBXli-_dCKCqAZpx ｜ ID not found in CD column
[8/10] ID: -ODQuu6Ej7h1hfCG476i ｜ ID not found in CD column
[9/10] ID: -ObWRZQcFgja8l7QuMyM ｜ ID not found in CD column
[10/10] ID: -NkEZ9LbyMsdggRl-HAK ｜ Y (3 cells)
=== All FO updates completed ===
